# Integración SQL y Python

**Seminario de Actualización — Unidad 5: Introducción a Data Analytics e Ingeniería de Datos**

## Objetivos de la clase

Al terminar esta práctica deberías poder:

1. Instalar y verificar el controlador ODBC necesario para conectar Python con SQL Server.
2. Crear una conexión con `pyodbc` y un *engine* de SQLAlchemy, sin escribir credenciales en el código.
3. Traer el resultado de una consulta a un `DataFrame` con `pandas.read_sql`, usando parámetros.
4. Escribir un `DataFrame` en una tabla de SQL Server con `to_sql`, entendiendo `if_exists`, `chunksize` y `fast_executemany`.

## Antes de empezar

**Primero el entorno virtual.** Como indica el *Anexo - Buenas Prácticas: Entornos Virtuales
con venv* del apunte, ninguna instalación de esta unidad debe hacerse sobre el Python global
del sistema. Desde la carpeta del proyecto:

```
python -m venv venv_seminario

venv_seminario\Scripts\activate.bat      # Windows (CMD)
venv_seminario\Scripts\Activate.ps1      # Windows (PowerShell)
source venv_seminario/bin/activate        # macOS / Linux
```

Con el entorno activado (el nombre `(venv_seminario)` aparece al inicio de la línea de
comandos), recién ahí se instalan las bibliotecas:

```
pip install pandas sqlalchemy pyodbc mssql-python ydata-profiling matplotlib seaborn jupyter
```

En Visual Studio Code hay que además **seleccionar el intérprete del entorno virtual**
(`Ctrl+Shift+P` → *Python: Select Interpreter*), o el notebook seguirá usando el Python global.

**Los dos controladores.** Esta unidad trabaja con `pyodbc` y con `mssql-python`, tal como los
presenta el apunte. `pyodbc` necesita además el **ODBC Driver 18 for SQL Server**, que no es un
paquete de Python sino un componente del sistema operativo y se descarga del sitio de Microsoft.
`mssql-python` no lo necesita: trae todo lo que hace falta en el propio paquete de pip.

> Este notebook está preparado para funcionar **aunque el servidor del laboratorio no esté
> disponible**: en ese caso trabaja con el conjunto sintético que genera `datos_demo.py`, que
> reproduce el esquema de Pampero (`ValoresPedido`, `Clientes`, catálogo de productos).
> Los ejercicios entregables, en cambio, deben resolverse contra la base real.

## 1. Verificación del entorno

La primera celda confirma que las bibliotecas están instaladas y, sobre todo, que el
sistema ve el controlador. Si la lista de controladores ODBC sale vacía o no incluye la
versión 18, el problema está en el sistema operativo y no en Python; con `mssql-python`,
en cambio, alcanza con que el paquete esté instalado.

In [ ]:
import sys
import pandas as pd

print("Python     :", sys.version.split()[0])
print("Intérprete :", sys.executable)   # debe apuntar al venv, no al Python global
print("pandas     :", pd.__version__)

try:
    import sqlalchemy
    print("SQLAlchemy :", sqlalchemy.__version__)
except ImportError:
    print("SQLAlchemy : NO INSTALADO  ->  pip install sqlalchemy")

try:
    import pyodbc
    print("pyodbc     :", pyodbc.version)
    print("\nControladores ODBC instalados:")
    for controlador in pyodbc.drivers():
        print("   -", controlador)
except ImportError:
    print("pyodbc     : NO INSTALADO  ->  pip install pyodbc")

try:
    import mssql_python
    print("\nmssql-python: instalado")
except ImportError:
    print("\nmssql-python: NO INSTALADO  ->  pip install mssql-python")

## 2. Conexión directa con pyodbc

`pyodbc` es la capa más baja: abre la conexión, ejecuta una sentencia y devuelve
tuplas de Python. Sirve para verificar que la conectividad funciona antes de sumar
SQLAlchemy y pandas encima.

Dos detalles de la cadena de conexión:

- **`Encrypt=yes`** es el valor por defecto desde el controlador 18. Es un cambio de postura
  de seguridad respecto de versiones anteriores.
- **`TrustServerCertificate=yes`** desactiva la validación del certificado del servidor. Se usa
  en el laboratorio porque el certificado es autofirmado; **en producción no debe usarse**.

In [ ]:
import config_conexion

print("Servidor configurado:", config_conexion.SERVIDOR)
print("Base configurada    :", config_conexion.BASE)
print("Autenticación       :",
      "SQL Server" if config_conexion.USUARIO else "Integrada de Windows")

# Prueba de conexión. Devuelve False (sin lanzar excepción) si no se puede conectar.
disponible = config_conexion.probar_conexion()
print("\n¿Servidor disponible?", disponible)

In [ ]:
# Conexión directa con pyodbc, tal como aparece en el apunte.
# Se ejecuta solo si el servidor responde.
if disponible:
    import pyodbc

    conn = pyodbc.connect(
        f"DRIVER={{{config_conexion.DRIVER}}};"
        f"SERVER={config_conexion.SERVIDOR};"
        f"DATABASE={config_conexion.BASE};"
        f"UID={config_conexion.USUARIO};"
        f"PWD={config_conexion.CLAVE};"
        "Encrypt=yes;TrustServerCertificate=yes;"
    )
    cursor = conn.cursor()
    cursor.execute("SELECT @@VERSION")
    print(cursor.fetchone()[0])

    cursor.execute("""
        SELECT TOP 10 TABLE_SCHEMA, TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME
    """)
    print("\nPrimeras tablas de la base:")
    for esquema, tabla in cursor.fetchall():
        print(f"   {esquema}.{tabla}")

    conn.close()
else:
    print("Servidor no disponible: se omite el bloque de pyodbc.")

## 3. SQLAlchemy: el enfoque recomendado con pandas

SQLAlchemy no reemplaza a `pyodbc`: lo usa por debajo. Lo que agrega es un objeto
`engine` que administra un *pool* de conexiones y una cadena de conexión uniforme,
independiente del motor de base de datos. Es lo que `pandas` espera recibir.

In [ ]:
print("Cadena de conexión que se va a usar:")
print("  ", config_conexion.cadena_conexion().replace(config_conexion.CLAVE or "\0", "***"))

if disponible:
    engine = config_conexion.crear_engine()
    print("\nEngine creado:", engine)
else:
    engine = None
    print("\nSin engine: el servidor no responde.")

## 4. Novedad: conexión con `mssql-python`

`mssql-python` es el controlador oficial de Microsoft para SQL Server, Azure SQL Database,
Azure SQL Managed Instance y SQL Database en Microsoft Fabric. Cumple la especificación
DB-API 2.0 y usa *Direct Database Connectivity* (DDBC), por lo que **no depende de un
administrador ODBC externo**: en Windows, la instalación con pip incluye todo lo necesario.
Requiere Python 3.10 o posterior.

La diferencia práctica con `pyodbc` es la instalación, no la forma de usarlo: la API de alto
nivel es la misma, y una vez creado el `engine`, `read_sql` y `to_sql` funcionan igual.

In [ ]:
# Conexión directa con mssql-python, sin SQLAlchemy de por medio.
# El administrador de contexto (with) cierra la conexión incluso si hay una excepción.
try:
    import mssql_python

    cadena = (
        f"Server={config_conexion.SERVIDOR};Database={config_conexion.BASE};"
        f"UID={config_conexion.USUARIO};PWD={config_conexion.CLAVE};"
        "Encrypt=yes;TrustServerCertificate=yes;"
    )

    with mssql_python.connect(cadena) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT TOP 5 * FROM Pedidos")
        for fila in cursor.fetchall():
            print(fila)

except ImportError:
    print("mssql-python no está instalado: pip install mssql-python")
except Exception as error:
    print(f"No se pudo conectar con mssql-python: {type(error).__name__}: {error}")

### Uso con SQLAlchemy

Desde **SQLAlchemy 2.1.0b2** existe un dialecto incorporado para `mssql-python`, con el
esquema de URL `mssql+mssqlpython`.

Al momento de escribir esto, **SQLAlchemy 2.1 sigue siendo una serie preliminar**: sirve para
laboratorio y evaluación, pero para producción conviene mantener SQLAlchemy 2.0.x con `pyodbc`,
fijar una versión concreta y probar la carga de trabajo antes de migrar. Por eso
`config_conexion.py` usa `pyodbc` por defecto y `mssql-python` solo si se lo pide
explícitamente:

```
$env:U05_CONTROLADOR = "mssqlpython"     # PowerShell
set U05_CONTROLADOR=mssqlpython          # CMD
export U05_CONTROLADOR=mssqlpython       # macOS / Linux
```

In [ ]:
# Las dos cadenas de conexión, una al lado de la otra.
print("pyodbc      :", config_conexion.cadena_conexion("pyodbc"))
print()
print("mssql-python:", config_conexion.cadena_conexion("mssqlpython"))
print()
config_conexion.resumen()

In [ ]:
# El mismo engine, con el dialecto nuevo. Requiere sqlalchemy >= 2.1.0b2.
#     pip install mssql-python "sqlalchemy>=2.1.0b2"
try:
    from sqlalchemy import create_engine, text

    engine_nuevo = create_engine(config_conexion.cadena_conexion("mssqlpython"))
    with engine_nuevo.connect() as conn:
        for fila in conn.execute(text("SELECT TOP 5 * FROM Pedidos")):
            print(fila)

except Exception as error:
    print(f"No disponible en este entorno: {type(error).__name__}: {error}")
    print("Es esperable si SQLAlchemy es 2.0.x o si falta mssql-python.")

Para Azure SQL con Microsoft Entra la URL toma la forma
`mssql+mssqlpython://@servidor.database.windows.net/base?authentication=ActiveDirectoryDefault&encrypt=yes`.
Es la forma recomendada en entornos productivos: no hay contraseña que administrar.

Una vez creado el `engine`, se reutiliza con SQLAlchemy Core, con el ORM y con las funciones
de pandas que aceptan una conexión de SQLAlchemy, como `read_sql` y `to_sql`. **Lo que cambia
es el dialecto y el controlador subyacente, no la API de alto nivel.**

## 5. De SQL a pandas: `read_sql`

`pandas.read_sql` ejecuta la consulta y devuelve directamente un `DataFrame`, infiriendo
los tipos de Python a partir de los tipos de columna de SQL Server.

**Regla no negociable:** los valores variables van en el parámetro `params`, nunca
concatenados dentro del texto de la consulta. Una consulta armada con f-strings es tan
vulnerable a inyección SQL desde Python como desde cualquier otro lenguaje cliente
(Unidad 4).

In [ ]:
# ✗ INCORRECTO — vulnerable a inyección SQL
# fecha = input("Fecha desde: ")
# df = pd.read_sql(f"SELECT * FROM ValoresPedido WHERE FechaPedido >= '{fecha}'", engine)

# ✓ CORRECTO — la consulta es fija; el valor viaja aparte
consulta = """
    SELECT IDPedido, FechaPedido, IDCliente, cantidad, val
    FROM ValoresPedido
    WHERE FechaPedido >= ?
"""

if disponible:
    fecha = input("Fecha desde: ")
    df = pd.read_sql(consulta, engine, params=(fecha,))
    print(df.head())
    print("\nFilas traídas:", len(df))
else:
    print("Servidor no disponible: se usará el conjunto sintético en la celda siguiente.")

In [ ]:
# ---------------------------------------------------------------------
# Conjunto de trabajo: la vista ValoresPedido unida a Clientes.
#
# ValoresPedido tiene IDPedido, IDCliente, IDEmpleado, EnvioPor, FechaPedido,
# FechaRequerida, FechaEnvio, cantidad y val. NO tiene Region, Pais ni Ciudad:
# esas columnas están en Clientes y hay que traerlas con un JOIN.
# ---------------------------------------------------------------------
import pandas as pd
import config_conexion
import datos_demo

CONSULTA = """
    SELECT v.IDPedido, v.IDCliente, v.IDEmpleado, v.EnvioPor,
           v.FechaPedido, v.FechaRequerida, v.FechaEnvio,
           v.cantidad, v.val,
           c.NombreEmpresa, c.Ciudad, c.Region, c.Pais
    FROM ValoresPedido AS v
      JOIN Clientes AS c
        ON c.IDCliente = v.IDCliente
"""

disponible = config_conexion.probar_conexion()

if disponible:
    engine = config_conexion.crear_engine()
    df = pd.read_sql(CONSULTA, engine)
    ORIGEN = "SQL Server"
else:
    engine = None
    tablas = datos_demo.generar_todo()
    df = tablas["ValoresPedido"].merge(
        tablas["Clientes"][["IDCliente", "NombreEmpresa", "Ciudad", "Region", "Pais"]],
        on="IDCliente", how="inner")
    ORIGEN = "datos sintéticos (datos_demo.py)"

print(f"Origen de los datos: {ORIGEN}")
print(f"Filas: {len(df)}   Columnas: {df.shape[1]}")
print(f"Columnas: {list(df.columns)}")

In [ ]:
df.head(10)

## 6. Agregar una columna derivada

Una columna derivada se calcula a partir de una o más columnas existentes. La forma más
directa es escribir el nombre de la nueva columna entre corchetes a la izquierda de la
asignación, y a la derecha la expresión que produce sus valores. **La operación se aplica a
todas las filas de una sola vez: no hace falta recorrer el `DataFrame` con un bucle.**

In [ ]:
# Cálculo a partir de una columna existente
df["MontoConIVA"] = (df["val"] * 1.21).round(2)

# Cálculo a partir de varias columnas: importe promedio por unidad vendida.
df["ValorPorUnidad"] = (df["val"] / df["cantidad"]).round(2)

df[["IDPedido", "cantidad", "val", "MontoConIVA", "ValorPorUnidad"]].head()

Tres cuidados con esta operación:

- Los nombres de la expresión deben **coincidir exactamente** con las columnas del `DataFrame`.
- Si algún operando tiene un valor faltante, el resultado de esa fila también será faltante,
  salvo que se lo trate antes con `fillna`.
- Asignar sobre un nombre ya existente **reemplaza** esa columna. Conviene elegir un nombre
  descriptivo y revisar el resultado con `head()` y `dtypes`.

In [ ]:
# El faltante se propaga: donde val es NaN, MontoConIVA también lo es.
print("Nulos en val         :", df["val"].isna().sum())
print("Nulos en MontoConIVA :", df["MontoConIVA"].isna().sum())
print()
print(df.dtypes)

In [ ]:
# Alternativa: assign() devuelve un DataFrame NUEVO con la columna agregada.
# Útil para encadenar transformaciones sin modificar el objeto original.
df_con_iva = df.assign(
    MontoConIVA=lambda datos: (datos["val"] * 1.21).round(2)
)

print("¿df quedó modificado?", "MontoConIVA" in df.columns)
print("Encadenado sobre una copia, sin tocar el original:")
(
    df.assign(FechaPedido=lambda d: pd.to_datetime(d["FechaPedido"], errors="coerce"))
      .assign(FechaEnvio=lambda d: pd.to_datetime(d["FechaEnvio"], errors="coerce"))
      .assign(DiasEnvio=lambda d: (d["FechaEnvio"] - d["FechaPedido"]).dt.days)
      [["IDPedido", "FechaPedido", "FechaEnvio", "DiasEnvio", "val"]]
      .head()
)

## 7. De pandas a SQL: `to_sql`

El camino inverso. El parámetro `if_exists` define qué hacer si la tabla destino ya existe:

| Valor | Comportamiento |
|---|---|
| `'fail'` | Aborta la operación (valor por defecto) |
| `'replace'` | Elimina y recrea la tabla |
| `'append'` | Agrega filas a la tabla existente |

Y dos parámetros de rendimiento que conviene recordar juntos:

- **`chunksize`**: envía los datos en lotes en lugar de una única transacción gigante.
- **`fast_executemany=True`** (en el `create_engine`): hace que el controlador **ODBC** agrupe
  los parámetros por lotes. La diferencia sobre miles de filas puede ser de un orden de magnitud.
  Es específico de `pyodbc`: con `mssql-python` no corresponde, por eso `config_conexion.py`
  solo lo aplica cuando el controlador efectivo es `pyodbc`.

In [ ]:
# Un resumen agregado, con una columna derivada calculada de forma vectorizada.
resumen = (
    df.assign(val=pd.to_numeric(df["val"], errors="coerce"))
      .groupby("Pais", as_index=False)
      .agg(Operaciones=("IDPedido", "count"),
           MontoTotal=("val", "sum"),
           MontoPromedio=("val", "mean"))
      .round(2)
)
resumen["MontoConIVA"] = (resumen["MontoTotal"] * 1.21).round(2)
resumen

In [ ]:
if disponible:
    resumen.to_sql(
        "VentasResumenU05",
        con=engine,
        if_exists="replace",   # 'fail' | 'replace' | 'append'
        index=False,
        chunksize=1000,
    )
    print("Tabla VentasResumenU05 escrita.")

    # Verificación: leer de vuelta lo que se acaba de escribir.
    print(pd.read_sql("SELECT * FROM VentasResumenU05 ORDER BY MontoTotal DESC", engine))
else:
    resumen.to_csv("ventas_resumen_u05.csv", index=False, encoding="utf-8")
    print("Sin servidor: el resumen se guardó en ventas_resumen_u05.csv")